In [1]:
# Setup: DuckDB — zero config, runs in-process, full SQL
# pip install duckdb
import duckdb

# Create an in-memory connection
con = duckdb.connect()

# -------------------------------------------------------
# Seed data: Citi-style server telemetry (employees for ranking demos)
# -------------------------------------------------------

def sql_executer(str):
    con.execute(str)

def sql_executer_printer(str):
    print (con.execute(str).df().to_string(index=False))   

In [2]:
sql = """
CREATE TABLE IF NOT EXISTS employees AS
SELECT * FROM (VALUES
    (1, 'Alice',   'Engineering', 120000),
    (2, 'Bob',     'Engineering', 120000),
    (3, 'Charlie', 'Engineering', 110000),
    (4, 'Diana',   'Engineering',  95000),
    (5, 'Eve',     'Marketing',   105000),
    (6, 'Frank',   'Marketing',   105000),
    (7, 'Grace',   'Marketing',    88000),
    (8, 'Henry',   'Finance',     115000),
    (9, 'Ivy',     'Finance',     115000),
    (10, 'Jack',   'Finance',      92000)
) t(id, name, department, salary)
"""
sql_executer(sql)
sql = """
CREATE TABLE IF NOT EXISTS server_metrics AS
SELECT * FROM (VALUES
    ('server-01', DATE '2026-01-01', 55.0),
    ('server-01', DATE '2026-01-02', 62.0),
    ('server-01', DATE '2026-01-03', 58.0),
    ('server-01', DATE '2026-01-04', 71.0),
    ('server-01', DATE '2026-01-05', 89.0),
    ('server-01', DATE '2026-01-06', 94.0),
    ('server-01', DATE '2026-01-07', 88.0),
    ('server-02', DATE '2026-01-01', 40.0),
    ('server-02', DATE '2026-01-02', 45.0),
    ('server-02', DATE '2026-01-03', 43.0),
    ('server-02', DATE '2026-01-04', 50.0),
    ('server-02', DATE '2026-01-05', 48.0),
    ('server-02', DATE '2026-01-06', 52.0),
    ('server-02', DATE '2026-01-07', 55.0)
) t(server_id, collection_date, cpu_utilization)
"""
sql_executer(sql)

sql="""
CREATE TABLE IF NOT Exists telemetry AS
SELECT * FROM (VALUES
    ('srv-01', '2026-03-01 08:00:00', 50.0, 'ACTIVE'),
    ('srv-01', '2026-03-01 16:00:00', 80.0, 'ACTIVE'),
    ('srv-01', '2026-03-02 08:00:00', 90.0, 'ACTIVE'),
    ('srv-01', '2026-03-02 16:00:00', 95.0, 'ACTIVE'),
    ('srv-02', '2026-03-01 10:00:00', 10.0, 'RETIRED'), -- Should be ignored!
    ('srv-03', '2026-03-01 08:00:00', 20.0, 'ACTIVE'),
    ('srv-03', '2026-03-01 16:00:00', 30.0, 'ACTIVE')
) t(server_id, collected_at, cpu_utilization, status)
"""

sql_executer(sql)

In [4]:
sql="""
WITH top_employees AS (
    SELECT id, name, salary
    FROM employees
    WHERE salary > 100000
)
SELECT *
FROM top_employees
ORDER BY salary DESC;
"""
sql_executer_printer(sql)

 id    name  salary
  1   Alice  120000
  2     Bob  120000
  8   Henry  115000
  9     Ivy  115000
  3 Charlie  110000
  5     Eve  105000
  6   Frank  105000


In [3]:
# Demonstrating basic CTE syntax structure
# -------------------------------------------------------
# A CTE is defined using the WITH keyword before the main query.

basic_cte_sql = """
WITH raw_telemetry AS (
    -- Step 1: Filter out the retired servers
    SELECT 
        server_id, 
        CAST(collected_at AS DATE) AS col_date, 
        cpu_utilization
    FROM telemetry
    WHERE status = 'ACTIVE'
),
daily_averages AS (
    -- Step 2: Average the CPU per server per day
    SELECT 
        server_id, 
        col_date, 
        AVG(cpu_utilization) AS avg_cpu
    FROM raw_telemetry
    GROUP BY server_id, col_date
)
-- Step 3: Find the highest daily average each server ever hit
SELECT 
    server_id, 
    MAX(avg_cpu) AS peak_daily_avg
FROM daily_averages
GROUP BY server_id
ORDER BY peak_daily_avg DESC;
"""

print("Syntax:")
print(basic_cte_sql)
sql_executer_printer(basic_cte_sql)

Syntax:

WITH raw_telemetry AS (
    -- Step 1: Filter out the retired servers
    SELECT 
        server_id, 
        CAST(collected_at AS DATE) AS col_date, 
        cpu_utilization
    FROM telemetry
    WHERE status = 'ACTIVE'
),
daily_averages AS (
    -- Step 2: Average the CPU per server per day
    SELECT 
        server_id, 
        col_date, 
        AVG(cpu_utilization) AS avg_cpu
    FROM raw_telemetry
    GROUP BY server_id, col_date
)
-- Step 3: Find the highest daily average each server ever hit
SELECT 
    server_id, 
    MAX(avg_cpu) AS peak_daily_avg
FROM daily_averages
GROUP BY server_id
ORDER BY peak_daily_avg DESC;

server_id  peak_daily_avg
   srv-01            92.5
   srv-03            25.0
